# Почасовая нормализация зарплат Avito

Повтор двух экспериментов: модели на приоритетных сегментах и схемы маршрутизации
для обычной работы, вахты и удалёнки. Объект прогноза — опубликованная точная сумма
или середина закрытой вилки. Все метрики в рублях за час на руки.
[Сводка и выводы](experiments/2026-11-18_salary_hourly/README.md)

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

cwd=Path.cwd().resolve()
root=next(p for p in (cwd,*cwd.parents) if (p/'ML/run_salary_hourly.py').is_file())
sys.path.insert(0,str(root/'ML'))
import run_salary_hourly as hourly

out=hourly.OUT
def read(name):
    return pd.read_csv(out/(name+'.csv'),dtype={'id':str})

support=read('hours_support')
group=['all','gold','кассир_спб','продавец_спб','комплектовщик_спб','ordinary','watch','remote']
cols=['scope','point_shift','exact','closed_range','native_hour','monthly_strict','gold_month_hour_overlap']
display(support.loc[support.scope.isin(group), cols])


,scope,point_shift,exact,closed_range,native_hour,monthly_strict,gold_month_hour_overlap
0,all,3054,86.0,1116,77,306,78
1,gold,785,23.0,359,13,105,78
3,ordinary,2281,68.0,667,57,277,78
4,watch,651,13.0,418,1,2,0
5,remote,122,5.0,31,19,27,0
8,кассир_спб,34,1.0,18,0,9,7
9,продавец_спб,107,1.0,79,0,19,17
10,комплектовщик_спб,97,0.0,35,1,9,5


## Два правила длительности и три способа сравнения

Строгий вариант делит ставку за смену на одно точное число часов.
Чувствительность рассчитывается на более широком наборе с серединой
одного закрытого диапазона, например 11,5 для 11–12 часов. Варианты
длительности, открытые границы, неизвестные часы и неизвестный период
остаются за пределами пересчёта. Налоговая база сохраняется.

В каждом фолде сравниваются прямой почасовой таргет и прогноз сменной
ставки с делением квантилей на часы вакансии. Эти две модели видят одни
и те же обучающие, валидационные и тестовые ID. Дополнительно пересчитаны
прежние отложенные прогнозы моделей, обученных на полном сменном пуле.
Для каждой части сохранено её исходное разбиение по работодателям.
В текстовой модели до построения словаря удаляются зарплатные выражения
и оставшиеся цифры.

In [2]:
metrics=read('metrics')
paired=read('paired')
cols=['scope','hours_policy','target','model','n','pinball','mae','coverage']
part=metrics.query("phase=='gold_models' and sample=='available' and scope in ['gold','кассир_спб','продавец_спб','комплектовщик_спб']")
display(part[cols].round(2))
display(paired.query("phase=='gold_models' and hours_policy=='single_range' and scope in ['gold','кассир_спб']")
        [['scope','model','comparison','n','delta','ci_low','ci_high']].round(2))


,scope,hours_policy,target,model,n,pinball,mae,coverage
21,gold,exact,direct_hour,cat_detail,23,104.31,227.42,0.22
24,кассир_спб,exact,direct_hour,cat_detail,1,10.67,2.22,1.00
25,продавец_спб,exact,direct_hour,cat_detail,1,389.19,858.59,0.00
27,gold,exact,direct_hour,chewby,23,112.38,239.16,0.43
30,кассир_спб,exact,direct_hour,chewby,1,27.08,43.75,1.00
...,...,...,...,...,...,...,...,...
168,комплектовщик_спб,single_range,shift_then_divide,empirical,35,163.92,330.97,0.49
170,gold,single_range,shift_then_divide,ssr_text,382,100.66,211.29,0.46
173,кассир_спб,single_range,shift_then_divide,ssr_text,19,20.13,49.32,0.47
174,продавец_спб,single_range,shift_then_divide,ssr_text,80,164.18,335.13,0.44


,scope,model,comparison,n,delta,ci_low,ci_high
91,gold,cat_detail,direct_hour vs shift_then_divide,382,3.26,0.07,6.97
92,gold,cat_detail,direct_hour vs previous_full_then_divide,382,2.18,-2.39,6.78
93,gold,chewby,direct_hour vs shift_then_divide,382,-0.19,-2.94,2.62
94,gold,chewby,direct_hour vs previous_full_then_divide,382,4.19,-0.42,9.03
95,gold,empirical,direct_hour vs shift_then_divide,382,2.74,-1.10,6.29
96,gold,empirical,direct_hour vs previous_full_then_divide,382,1.64,-2.80,5.23
97,gold,ssr_text,direct_hour vs shift_then_divide,382,2.74,-0.87,6.68
98,gold,ssr_text,direct_hour vs previous_full_then_divide,382,3.14,-1.39,8.26
99,gold,ssr_text,ssr_text vs chewby (direct_hour),382,-1.67,-4.47,1.16
100,gold,ssr_text,ssr_text vs chewby (shift_then_divide),382,-4.60,-7.64,-1.68


## Повтор маршрутизации

Сравниваем общий CatBoost, CatBoost по режимам, локальные модели для ячеек
внутри режимов, гибрид и режимные квартили. Ветка локальных моделей
сохраняет заданные ранее пороги опоры. Срез common_four содержит только ID
с прогнозом каждой из четырёх модельных схем в обеих новых постановках.
Если такой выборки нет, это ограничение поддержки данных.
Срез сумм до 50 000 ₽ на вахте использует исходную сумму за смену только
для диагностики ошибки. Расхождение между сменой и полной ротацией
не исправляется делением на часы.

In [3]:
coverage=read('coverage')
display(coverage.query("scope=='all' and target=='direct_hour'")[
        ['hours_policy','model','all_civilian','convertible_context','scored_context',
         'labelled','scored_labelled','coverage_all','coverage_labelled']].round(4))
display(metrics.query("phase=='routing' and scope=='all' and sample in ['available','common_four']")[
        ['sample',*cols]].round(2))
display(metrics.query("phase=='routing' and hours_policy=='single_range' and sample=='available' and scope in ['ordinary','watch','remote','watch_tail_le_50000']")
        [cols].round(2))
display(paired.query("phase=='extension' and scope=='all'").round(2))
display(metrics.query("phase=='extension' and scope=='all'")[
        ['source_unit','n','pinball','mae','coverage']].round(2))


,hours_policy,model,all_civilian,convertible_context,scored_context,labelled,scored_labelled,coverage_all,coverage_labelled
0,exact,by_mode,31674,204,177,86,68,0.0056,0.7907
10,exact,by_segment,31674,204,0,86,0,0.0000,0.0000
20,exact,empirical,31674,204,177,86,68,0.0056,0.7907
30,exact,global,31674,204,177,86,68,0.0056,0.7907
40,exact,hybrid,31674,204,177,86,68,0.0056,0.7907
100,single_range,by_mode,31674,2445,2445,1202,1202,0.0772,1.0000
110,single_range,by_segment,31674,2445,108,1202,62,0.0034,0.0516
120,single_range,empirical,31674,2445,2445,1202,1202,0.0772,1.0000
130,single_range,global,31674,2445,2445,1202,1202,0.0772,1.0000
140,single_range,hybrid,31674,2445,2445,1202,1202,0.0772,1.0000


,sample,scope,hours_policy,target,model,n,pinball,mae,coverage
176,available,all,exact,direct_hour,by_mode,68,97.98,216.36,0.22
186,available,all,exact,direct_hour,by_segment,0,NaN,NaN,NaN
196,available,all,exact,direct_hour,empirical,68,93.40,204.55,0.46
206,available,all,exact,direct_hour,global,68,85.37,188.60,0.46
216,available,all,exact,direct_hour,hybrid,68,85.37,188.60,0.46
226,available,all,exact,previous_full_then_divide,by_mode,86,322.92,674.58,0.51
236,available,all,exact,previous_full_then_divide,by_segment,14,77.92,160.66,0.43
246,available,all,exact,previous_full_then_divide,empirical,86,301.24,593.30,0.51
256,available,all,exact,previous_full_then_divide,global,86,336.49,705.38,0.44
266,available,all,exact,previous_full_then_divide,hybrid,86,336.39,704.93,0.44


,scope,hours_policy,target,model,n,pinball,mae,coverage
401,ordinary,single_range,direct_hour,by_mode,735,111.42,233.17,0.43
402,watch,single_range,direct_hour,by_mode,431,1119.56,2335.84,0.39
403,remote,single_range,direct_hour,by_mode,36,265.05,596.51,0.19
405,watch_tail_le_50000,single_range,direct_hour,by_mode,381,94.41,184.56,0.44
412,ordinary,single_range,direct_hour,by_segment,44,278.46,562.54,0.16
413,watch,single_range,direct_hour,by_segment,18,54.19,123.51,0.39
414,remote,single_range,direct_hour,by_segment,0,NaN,NaN,NaN
416,watch_tail_le_50000,single_range,direct_hour,by_segment,18,54.19,123.51,0.39
423,ordinary,single_range,direct_hour,empirical,735,119.20,248.82,0.46
424,watch,single_range,direct_hour,empirical,431,1094.63,2303.90,0.47


,phase,hours_policy,scope,model,comparison,n,groups,first_pinball,second_pinball,delta,ci_low,ci_high
377,extension,single_range,all,global,plus native hours vs shift-derived hours only,1202,676,484.78,484.81,-0.03,-5.72,5.22


,source_unit,n,pinball,mae,coverage
0,hour,77,281.97,591.69,0.38
9,shift,1202,484.78,995.64,0.49


## Применение без обучения

Используем архивные входные строки из прежнего исследования маршрутизации.
Почасовой пакет выбирается по политике длительности. Для каждой строки
показаны происхождение часов, прогноз и причина отказа.
[Пакеты и правила входа](models/2026-11-18_salary_hourly/README.md)

In [4]:
sample=pd.read_csv(hourly.r.EXPERIMENT_DIR/'example_vacancies.csv',dtype=str,keep_default_na=False)
preview=hourly.predict_hourly(sample,hours_policy='single_range')
shift_model={'version':1,'rules':[],'default':'global','fallback':['empirical']}
previous=hourly.r.predict_vacancies(sample,shift_model)
preview['p50_shift_model_divided']=(previous.p50 / preview.hours).where(preview.p50_hour.notna())
display(preview[['id','mode','unit','tax','hours_status','hours','rule','approach',
                 'p25_hour','p50_hour','p75_hour','p50_shift_model_divided','reason','limit']].round(2))

,id,mode,unit,tax,hours_status,hours,rule,approach,p25_hour,p50_hour,p75_hour,p50_shift_model_divided,reason,limit
0,2971866871,ordinary,shift,net,closed_range,11.5,default,global,270.89,317.62,430.54,345.41,conditional_model,published_offers
1,1012505611,ordinary,shift,net,closed_range,11.5,default,global,251.31,313.91,369.02,322.21,conditional_model,published_offers
2,4347741400,ordinary,shift,net,missing,NaN,default,abstain,NaN,NaN,NaN,NaN,unsupported_hours,published_offers
3,1317059545,watch,shift,net,closed_range,9.5,default,global,355.78,485.80,734.68,472.59,conditional_model,verify_shift_vs_rotation
4,2459477803,remote,shift,net,closed_range,11.5,default,global,311.43,383.53,462.24,263.10,conditional_model,published_offers
5,2526217995,remote,shift,net,closed_range,6.5,default,global,364.07,456.18,600.50,849.57,conditional_model,published_offers
6,1040408058,ordinary,unspecified,unknown,missing,NaN,default,abstain,NaN,NaN,NaN,NaN,shift_net_required,published_offers
7,1172455832,ordinary,shift,unknown,missing,NaN,default,abstain,NaN,NaN,NaN,NaN,shift_net_required,published_offers
8,1771751639,ordinary,hour,net,missing,NaN,default,abstain,NaN,NaN,NaN,NaN,shift_net_required,published_offers
9,4288354022,ordinary,shift,net,closed_range,11.5,default,global,304.14,357.51,417.77,365.01,conditional_model,published_offers
